# 01 — Data Collection

Pulling live ADS-B state vectors from the **OpenSky Network** REST API for a defined airspace bounding box, clean the data, and persist it to CSV for downstream notebooks.


## 1. Configuration

Dallas/Fort Worth TRACON (D10) is the default sector — one of the busiest in the US and well-documented for comparison. Swap the bounding box to target any airspace.

> #### Tracon, or **Terminal Radar Approach Control**, is an airport's FAA facility that manages aircraft within a 30-50 mile radius of the airport typically within 10-15 thousand feet.

In [1]:
import sys, os, time, json, requests
import pandas as pd
from datetime import datetime, timezone

sys.path.insert(0, os.path.abspath('../src'))
from airspace import parse_opensky_response, states_to_df

# ── Bounding box: Dallas/Fort Worth TRACON (D10) ──────────────────────────
BBOX = {
    "lamin": 32.0,   # min latitude
    "lamax": 33.5,   # max latitude
    "lomin": -98.0,  # min longitude
    "lomax": -96.0,  # max longitude
}

'''
Example Bounding box for Chicago O'Hare TRACON (C90):
BBOX = {
    "lamin": 39.5,   # min latitude
    "lamax": 42.5,   # max latitude
    "lomin": -88.5,  # min longitude
    "lomax": -87.0,  # max longitude
}
'''

NUM_SNAPSHOTS = 6      # snapshots to collect
SLEEP_SEC   = 10     # OpenSky free tier: 1 req/10s, need to wait between snapshots
OUTPUT_PATH = "../data/raw_states.csv"

print(f"Bounding box: {BBOX}")
print(f"Collecting {NUM_SNAPSHOTS} snapshots (~{NUM_SNAPSHOTS * SLEEP_SEC}s total)...")

Bounding box: {'lamin': 32.0, 'lamax': 33.5, 'lomin': -98.0, 'lomax': -96.0}


#### 1.1 BBOX locator

The cell above uses bbox for defining TRACON locations. You can use the function below to find a bbox for any airport if you know it's icao code.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
from get_bbox import main as get_bbox

get_bbox()


Bounding box for KJFK:
lamin: 40.6210509,
lamax: 40.6647615,
lomin: -73.8232337,
lomax: -73.7483913


## 2. Fetch from OpenSky

The `/states/all` endpoint returns every aircraft currently visible within the bounding box as a list of state vectors.

In [4]:
BASE_URL  = "https://opensky-network.org/api/states/all"
all_states, failed = [], 0

for i in range(NUM_SNAPSHOTS):
    try:
        resp = requests.get(BASE_URL, params=BBOX, timeout=15)
        resp.raise_for_status()
        data     = resp.json()
        snapshot = parse_opensky_response(data)
        all_states.extend(snapshot)
        ts = datetime.fromtimestamp(data.get("time", 0), tz=timezone.utc)
        print(f"  [{i+1}/{NUM_SNAPSHOTS}] {ts.strftime('%H:%M:%S')} UTC — {len(snapshot)} aircraft")
    except Exception as e:
        print(f"  [{i+1}/{NUM_SNAPSHOTS}] FAILED: {e}")
        failed += 1
    if i < NUM_SNAPSHOTS - 1:
        time.sleep(SLEEP_SEC)

print(f"\nTotal state vectors collected: {len(all_states)}  ({failed} snapshots failed)")


  [1/6] 04:22:13 UTC — 28 aircraft
  [2/6] 04:22:28 UTC — 28 aircraft
  [3/6] 04:22:39 UTC — 28 aircraft
  [4/6] 04:22:49 UTC — 29 aircraft
  [5/6] 04:22:57 UTC — 27 aircraft
  [6/6] 04:22:54 UTC — 28 aircraft

Total state vectors collected: 168  (0 snapshots failed)


## 3. Clean & Validate

Drop records with missing critical fields, filter out ground traffic, and apply sanity bounds.

In [ ]:
df = states_to_df(all_states)
print(f"Raw shape: {df.shape}")
print("\nNull counts per column:")
print(df.isnull().sum())

Raw shape: (310, 9)

Null counts per column:
icao24          0
callsign        0
time            0
lat             0
lon             0
altitude_ft     0
velocity_kts    0
heading_deg     0
on_ground       0
dtype: int64


In [ ]:
df_clean = (df
    .dropna(subset=["lat", "lon", "altitude_ft", "velocity_kts"])
    .query("not on_ground")
    .query("0 < altitude_ft < 60000")
    .query("0 < velocity_kts < 700")
    .query("@BBOX['lamin'] <= lat <= @BBOX['lamax']")
    .query("@BBOX['lomin'] <= lon <= @BBOX['lomax']")
    .copy()
)

df_clean["callsign"]  = df_clean["callsign"].str.strip().replace("", "UNKNOWN")
df_clean["time_utc"]  = pd.to_datetime(df_clean["time"], unit="s", utc=True)
df_clean["alt_band"]  = pd.cut(df_clean["altitude_ft"],
                                bins=[0, 10000, 18000, 28000, 60000],
                                labels=["Low", "Terminal", "Transition", "High"])

print(f"Clean shape: {df_clean.shape}  ({len(df) - len(df_clean)} rows dropped)")
df_clean.head(10)

Clean shape: (310, 11)  (0 rows dropped)


,icao24,callsign,time,lat,lon,altitude_ft,velocity_kts,heading_deg,on_ground,time_utc,alt_band
0,a33b23,UPS765,1.773283e+09,33.1031,-97.0117,10600.000339,279.76104,24.27,False,2026-03-12 02:39:57+00:00,Terminal
1,adda24,AAL1456,1.773283e+09,32.8118,-97.0551,1550.000050,120.02256,0.48,False,2026-03-12 02:39:57+00:00,Low
2,a76d5a,JIA5497,1.773283e+09,32.6071,-97.0171,3725.000119,175.91256,331.10,False,2026-03-12 02:39:57+00:00,Low
3,ad4390,AAL2749,1.773283e+09,32.9872,-97.0277,3675.000118,225.07632,1.27,False,2026-03-12 02:39:57+00:00,Low
4,a75b65,AAL3082,1.773283e+09,32.4976,-97.3005,5275.000169,234.58248,83.39,False,2026-03-12 02:39:57+00:00,Low
5,a8b926,NKS1496,1.773283e+09,32.8877,-96.9923,400.000013,143.56440,314.72,False,2026-03-12 02:39:57+00:00,Low
6,a6ae2f,N53BB,1.773283e+09,32.7516,-96.5305,450.000014,101.01024,0.57,False,2026-03-12 02:39:57+00:00,Low
7,ab7627,AAL2987,1.773283e+09,32.6952,-97.0558,2725.000087,154.02312,0.74,False,2026-03-12 02:39:57+00:00,Low
8,a353c6,UPS5772,1.773283e+09,32.9857,-97.2679,9975.000319,307.28808,246.61,False,2026-03-12 02:39:57+00:00,Low
9,c07b8c,WJA2171,1.773283e+09,32.7967,-97.3917,36000.001152,424.04472,322.86,False,2026-03-12 02:39:57+00:00,High


## 4. Save

In [ ]:
os.makedirs("../data", exist_ok=True)
df_clean.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(df_clean)} records -> {OUTPUT_PATH}")
print(f"\nAltitude band distribution:")
print(df_clean["alt_band"].value_counts())

Saved 310 records -> ../data/raw_states.csv

Altitude band distribution:
alt_band
Low           202
Terminal       75
Transition     21
High           12
Name: count, dtype: int64
